# Imports

In [1]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Ablation: alpha < 1, rank ratio = 1 || alpha = 1, rank ratio <=1, random projection

## load data

In [2]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/diff_trans'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'seq_len', 'pred_len', 'data_id', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'batch_size', 'lradj', 'patience', 'train_epochs']
metric_names = ['mse', 'mae']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    result.loc[:, metric_names] = metric[1], metric[0]
    df.append(result)

df = pd.concat(df, ignore_index=True)
df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df.head(4)

,model,seq_len,pred_len,data_id,learning_rate,rec_lambda,auxi_lambda,pca_dim,reinit,use_weights,rank_ratio,auxi_loss,batch_size,lradj,patience,train_epochs,mse,mae
77,FreTS,96,96,Weather_FA,0.001,0.2,0.8,T,1,0,1.0,MAE,32,type1,3,10,0.171692,0.224669
44,FreTS,96,192,Weather_FA,0.001,0.2,0.8,T,1,0,1.0,MAE,32,type1,3,10,0.212493,0.263629
89,FreTS,96,336,Weather_FA,0.001,0.2,0.8,T,1,0,1.0,MAE,32,type1,3,10,0.261259,0.299690
300,FreTS,96,720,Weather_FA,0.001,0.2,0.8,T,1,0,1.0,MAE,32,type1,3,10,0.335380,0.360984


## load data rebb

In [3]:
root = '/data/home/Licheng/workspace/TSF-PCA/results_PCA/diff_trans_rebb'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'seq_len', 'pred_len', 'data_id', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'pca_dim', 'reinit', 'use_weights', 'rank_ratio', 'auxi_loss', 'batch_size', 'lradj', 'patience', 'train_epochs']
metric_names = ['mse', 'mae']

df_rebb = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    result.loc[:, metric_names] = metric[1], metric[0]
    df_rebb.append(result)

df_rebb = pd.concat(df_rebb, ignore_index=True)
df_rebb.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df_rebb.head(4)

,model,seq_len,pred_len,data_id,learning_rate,rec_lambda,auxi_lambda,pca_dim,reinit,use_weights,rank_ratio,auxi_loss,batch_size,lradj,patience,train_epochs,mse,mae
34,FreTS,96,96,Weather_Random,0.0004,0.0,1.0,T,1,0,1.0,MAE,32,type1,3,10,0.169945,0.214964
119,FreTS,96,96,Weather_Random,0.0003,0.0,1.0,T,1,0,1.0,MAE,32,type1,3,10,0.171654,0.216642
177,FreTS,96,96,Weather_Random,0.0002,0.0,1.0,T,1,0,1.0,MAE,32,type1,3,10,0.175259,0.220235
305,FreTS,96,96,Weather_Random,0.0020,0.0,1.0,T,1,0,1.0,MAE,32,type1,3,10,0.168404,0.212037


## preprocess

In [4]:

save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'

baselines = pd.read_csv(f'{save_root}/baselines_params.csv')
finetunes_all = pd.read_csv(f'{save_root}/finetune_all_results.csv')

base = baselines.copy()
base = base[
    ((base.data_id == 'ETTh1') & (base.model == 'Fredformer')) |
    ((base.data_id == 'ETTh2') & (base.model == 'Fredformer')) |
    ((base.data_id == 'ETTm1') & (base.model == 'Fredformer')) |
    ((base.data_id == 'ETTm2') & (base.model == 'Fredformer')) |
    ((base.data_id == 'ECL') & (base.model == 'iTransformer')) |
    ((base.data_id == 'Traffic') & (base.model == 'iTransformer')) |
    ((base.data_id == 'Weather') & (base.model == 'FreTS')) |
    ((base.data_id == 'PEMS03') & (base.model == 'MICN')) |
    ((base.data_id == 'PEMS08') & (base.model == 'iTransformer'))
]

aba1 = finetunes_all.copy()
aba1 = aba1[
    (aba1.pca_dim == 'T') &
    (aba1.use_weights == 0) &
    (aba1.auxi_loss == 'MAE') &
    (aba1.reinit == 1) &
    (aba1.seq_len == 96) &
    (aba1.rank_ratio == 1.0)
]
aba1 = aba1[
    ((aba1.data_id == 'ETTh1_PCA') & (aba1.model == 'Fredformer')) |
    ((aba1.data_id == 'ETTh2_PCA') & (aba1.model == 'Fredformer')) |
    ((aba1.data_id == 'ETTm1_PCA') & (aba1.model == 'Fredformer')) |
    ((aba1.data_id == 'ETTm2_PCA') & (aba1.model == 'Fredformer')) |
    ((aba1.data_id == 'ECL_PCA') & (aba1.model == 'iTransformer')) |
    ((aba1.data_id == 'Traffic_PCA') & (aba1.model == 'iTransformer')) |
    ((aba1.data_id == 'Weather_PCA') & (aba1.model == 'FreTS')) |
    ((aba1.data_id == 'PEMS03_PCA') & (aba1.model == 'MICN')) |
    ((aba1.data_id == 'PEMS08_PCA') & (aba1.model == 'iTransformer'))
]
aba1.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
aba1.drop(columns=['rec_lambda'], inplace=True)


aba2 = df.copy()
aba2 = aba2[
    (aba2.pca_dim == 'T') &
    (aba2.use_weights == 0) &
    (aba2.auxi_loss == 'MAE') &
    (aba2.reinit == 1) &
    (aba2.seq_len == 96) &
    (aba2.rank_ratio < 1.0) &
    (aba2.learning_rate.isin([0.0005]))
]
aba2 = aba2[
    ((aba2.data_id == 'ETTh1_Random') & (aba2.model == 'Fredformer')) |
    ((aba2.data_id == 'ETTh2_Random') & (aba2.model == 'Fredformer')) |
    ((aba2.data_id == 'ETTm1_Random') & (aba2.model == 'Fredformer')) |
    ((aba2.data_id == 'ETTm2_Random') & (aba2.model == 'Fredformer')) |
    ((aba2.data_id == 'ECL_Random') & (aba2.model == 'iTransformer')) |
    ((aba2.data_id == 'Traffic_Random') & (aba2.model == 'iTransformer')) |
    ((aba2.data_id == 'Weather_Random') & (aba2.model == 'FreTS')) |
    ((aba2.data_id == 'PEMS03_Random') & (aba2.model == 'MICN')) |
    ((aba2.data_id == 'PEMS08_Random') & (aba2.model == 'iTransformer'))
]
aba2_rebb = df_rebb.copy()
aba2_rebb = aba2_rebb[
    (aba2_rebb.pca_dim == 'T') &
    (aba2_rebb.use_weights == 0) &
    (aba2_rebb.auxi_loss == 'MAE') &
    (aba2_rebb.reinit == 1) &
    (aba2_rebb.seq_len == 96) &
    (aba2_rebb.rank_ratio < 1.0) &
    (aba2_rebb.learning_rate.isin([0.0005]))
]
aba2_rebb = aba2_rebb[aba2.columns]
aba2 = pd.concat([aba2, aba2_rebb], ignore_index=True)
aba2.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
aba2.drop(columns=['rec_lambda'], inplace=True)


aba21 = df.copy()
aba21 = aba21[
    (aba21.pca_dim == 'T') &
    (aba21.use_weights == 0) &
    (aba21.auxi_loss == 'MAE') &
    (aba21.reinit == 1) &
    (aba21.seq_len == 96) &
    (aba21.rank_ratio == 1.0)
]
aba21 = aba21[
    ((aba21.data_id == 'ETTh1_Random') & (aba21.model == 'Fredformer')) |
    ((aba21.data_id == 'ETTh2_Random') & (aba21.model == 'Fredformer')) |
    ((aba21.data_id == 'ETTm1_Random') & (aba21.model == 'Fredformer')) |
    ((aba21.data_id == 'ETTm2_Random') & (aba21.model == 'Fredformer')) |
    ((aba21.data_id == 'ECL_Random') & (aba21.model == 'iTransformer')) |
    ((aba21.data_id == 'Traffic_Random') & (aba21.model == 'iTransformer')) |
    ((aba21.data_id == 'Weather_Random') & (aba21.model == 'FreTS')) |
    ((aba21.data_id == 'PEMS03_Random') & (aba21.model == 'MICN')) |
    ((aba21.data_id == 'PEMS08_Random') & (aba21.model == 'iTransformer'))
]
aba21_rebb = df_rebb.copy()
aba21_rebb = aba21_rebb[
    (aba21_rebb.pca_dim == 'T') &
    (aba21_rebb.use_weights == 0) &
    (aba21_rebb.auxi_loss == 'MAE') &
    (aba21_rebb.reinit == 1) &
    (aba21_rebb.seq_len == 96) &
    (aba21_rebb.rank_ratio == 1.0)
]
aba21_rebb = aba21_rebb[aba21.columns]
aba21 = pd.concat([aba21, aba21_rebb], ignore_index=True)
aba21.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
aba21.drop(columns=['rec_lambda'], inplace=True)



aba3 = finetunes_all.copy()
aba3 = aba3[
    (aba3.pca_dim == 'T') &
    (aba3.use_weights == 0) &
    (aba3.auxi_loss == 'MAE') &
    (aba3.reinit == 1) &
    (aba3.seq_len == 96)
]
aba3 = aba3[
    ((aba3.data_id == 'ETTh1_PCA') & (aba3.model == 'Fredformer')) |
    ((aba3.data_id == 'ETTh2_PCA') & (aba3.model == 'Fredformer')) |
    ((aba3.data_id == 'ETTm1_PCA') & (aba3.model == 'Fredformer')) |
    ((aba3.data_id == 'ETTm2_PCA') & (aba3.model == 'Fredformer')) |
    ((aba3.data_id == 'ECL_PCA') & (aba3.model == 'iTransformer')) |
    ((aba3.data_id == 'Traffic_PCA') & (aba3.model == 'iTransformer')) |
    ((aba3.data_id == 'Weather_PCA') & (aba3.model == 'FreTS')) |
    ((aba3.data_id == 'PEMS03_PCA') & (aba3.model == 'MICN')) |
    ((aba3.data_id == 'PEMS08_PCA') & (aba3.model == 'iTransformer'))
]
aba3.rename(columns={'auxi_lambda': 'alpha'}, inplace=True)
aba3.drop(columns=['rec_lambda'], inplace=True)


In [ ]:
print(len(aba1.data_id.unique()), aba1.data_id.unique())
print(len(aba2.data_id.unique()), aba2.data_id.unique())
print(len(aba21.data_id.unique()), aba21.data_id.unique())
print(len(aba3.data_id.unique()), aba3.data_id.unique())


In [ ]:

data_list = ['ECL_PCA']
tmp = aba3[aba3.data_id.isin(data_list)].copy()

# data_list = ['ETTh1_Random']
# tmp = aba2[aba2.data_id.isin(data_list)].copy()

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
min_mse_idx = tmp.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
tmp = tmp.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
tmp = tmp[columns]

dst_order = data_list
tmp['data_id'] = pd.Categorical(tmp['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
tmp['model'] = pd.Categorical(tmp['model'], categories=model_order, ordered=True)

tmp_avg = tmp.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
tmp_avg['pred_len'] = 'Avg'

tmp = pd.concat([tmp, tmp_avg]).reset_index(drop=True)
tmp.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)

tmp.dropna(inplace=True, thresh=5)

# tmp['data_id'] = tmp['data_id'].str.replace('_PCA', '', regex=False)
# tmp = tmp[['model', 'pred_len', 'data_id', 'mse', 'mae']]
# tmp['label'] = r'PDF$^\ddagger$'
tmp

## analysis base

In [5]:
df1 = base.copy()

df1 = df1[['model', 'pred_len', 'data_id', 'mse', 'mae']]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
df1['data_id'] = pd.Categorical(df1['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df1['model'] = pd.Categorical(df1['model'], categories=model_order, ordered=True)

df1_avg = df1.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df1_avg['pred_len'] = 'Avg'
df1 = pd.concat([df1, df1_avg]).reset_index(drop=True)

df1.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

df1.dropna(inplace=True, thresh=4)

df1['label'] = 'DF'
df1.head(5)

/tmp/ipykernel_1686659/1829447716.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df1_avg = df1.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,label
12,Fredformer,96,ETTm1,0.326369,0.360869,DF
13,Fredformer,192,ETTm1,0.365194,0.382132,DF
14,Fredformer,336,ETTm1,0.395987,0.404369,DF
15,Fredformer,720,ETTm1,0.459217,0.444342,DF
36,Fredformer,Avg,ETTm1,0.386692,0.397928,DF


## analysis ablation 1

In [ ]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
# aba1[(aba1['pred_len'] == 192) & (aba1['data_id'] == 'ETTm1_PCA')].sort_values(by=['pred_len', 'mse'])[columns].head(10)
# aba1[(aba1['pred_len'] == 336) & (aba1['data_id'] == 'ETTh1_PCA')].sort_values(by=['pred_len', 'mse'])[columns].head(10)
# aba1[(aba1['pred_len'] == 336) & (aba1['data_id'] == 'ECL_PCA')].sort_values(by=['pred_len', 'mse'])[columns].head(10)
# aba1[(aba1['pred_len'] == 720) & (aba1['data_id'] == 'ETTm2_PCA')].sort_values(by=['pred_len', 'mse'])[columns].head(10)
# aba1[(aba1['pred_len'] == 720) & (aba1['data_id'] == 'ETTh2_PCA')].sort_values(by=['pred_len', 'mse'])[columns].head(10)
aba1[(aba1['pred_len'] == 720) & (aba1['data_id'] == 'Traffic_PCA')].sort_values(by=['pred_len', 'mse'])[columns].head(10)
# aba1[(aba1['pred_len'] == 12) & (aba1['data_id'] == 'PEMS08_PCA')].sort_values(by=['pred_len', 'mse'])[columns].head(10)

In [ ]:
df2 = aba1.copy()
df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'Traffic_PCA') & (df2.learning_rate == 0.001) & (df2.alpha == 0.6)]

In [6]:
min_mode = 'each'

df2 = aba1.copy()
df2_m1_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTm1_PCA') & (df2.learning_rate == 0.0005) & (df2.alpha == 0.9)]
df2_m1_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ETTm1_PCA') & (df2.learning_rate == 0.0005) & (df2.alpha == 0.7)]

df2_m2_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTm2_PCA') & (df2.learning_rate == 0.0001) & (df2.alpha == 1.0)]
df2_m2_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ETTm2_PCA') & (df2.learning_rate == 0.0001) & (df2.alpha == 1.0)]
df2_m2_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ETTm2_PCA') & (df2.learning_rate == 0.0005) & (df2.alpha == 1.0)]
df2_m2_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'ETTm2_PCA') & (df2.learning_rate == 0.0005) & (df2.alpha == 0.9)]

df2_h1_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTh1_PCA') & (df2.learning_rate == 0.0005) & (df2.alpha == 0.9)]
df2_h1_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ETTh1_PCA') & (df2.learning_rate == 0.00005) & (df2.alpha == 0.2)]
df2_h1_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ETTh1_PCA') & (df2.learning_rate == 0.0005) & (df2.alpha == 0.9)]

df2_h2_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTh2_PCA') & (df2.learning_rate == 0.0005) & (df2.alpha == 0.9)]
df2_h2_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'ETTh2_PCA') & (df2.learning_rate == 0.00005) & (df2.alpha == 1.0)]

df2_ecl_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ECL_PCA') & (df2.learning_rate == 0.001) & (df2.alpha == 0.1)]
df2_ecl_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ECL_PCA') & (df2.learning_rate == 0.001) & (df2.alpha == 0.5)]
df2_ecl_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ECL_PCA') & (df2.learning_rate == 0.001) & (df2.alpha == 0.7)]

df2_tra_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'Traffic_PCA') & (df2.learning_rate == 0.001) & (df2.alpha == 0.6) & (df2.batch_size == 8)]
df2_tra_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'Traffic_PCA') & (df2.learning_rate == 0.001) & (df2.alpha == 0.6) & (df2.batch_size == 8)]
df2_tra_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'Traffic_PCA') & (df2.learning_rate == 0.001) & (df2.alpha == 0.6) & (df2.batch_size == 8)]
df2_tra_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'Traffic_PCA') & (df2.learning_rate == 0.001) & (df2.alpha == 0.2) & (df2.batch_size == 8)]

df2_p8_12 = df2[(df2['pred_len'] == 12) & (df2['data_id'] == 'PEMS08_PCA') & (df2.learning_rate == 0.001) & (df2.alpha == 1.0)]

df2_other = df2[
    ((df2['data_id'] == 'ETTm1_PCA') & (df2['pred_len'].isin([336, 720]))) |
    ((df2['data_id'] == 'ETTh1_PCA') & (df2['pred_len'].isin([720]))) |
    ((df2['data_id'] == 'ECL_PCA') & (df2['pred_len'].isin([720]))) |
    ((df2['data_id'] == 'ETTh2_PCA') & (df2['pred_len'].isin([192, 336]))) |
    ((df2['data_id'] == 'PEMS08_PCA') & (df2['pred_len'].isin([24, 36, 48]))) |
    df2.data_id.isin(['Weather_PCA', 'PEMS03_PCA'])
]
df2 = pd.concat([
    df2_m1_96, df2_m1_192, 
    df2_m2_96, df2_m2_192, df2_m2_336, df2_m2_720,
    df2_h1_96, df2_h1_192, df2_h1_336,
    df2_h2_96, df2_h2_720,
    df2_ecl_96, df2_ecl_192, df2_ecl_336,
    df2_tra_96, df2_tra_192, df2_tra_336, df2_tra_720,
    df2_p8_12,
    df2_other
], ignore_index=True)

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
if min_mode == 'group':
    # df2 = df2.groupby(columns).filter(is_full_group)
    mse_mean = df2.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df2 = df2.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df2 = df2.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
df2 = df2[columns]

dst_order = ['ETTm1_PCA', 'ETTm2_PCA', 'ETTh1_PCA', 'ETTh2_PCA', 'ECL_PCA', 'Traffic_PCA', 'Weather_PCA', 'PEMS03_PCA', 'PEMS08_PCA']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df2['model'] = pd.Categorical(df2['model'], categories=model_order, ordered=True)

df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)
df2.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)

df2.dropna(inplace=True, thresh=5)

df2['data_id'] = df2['data_id'].str.replace('_PCA', '', regex=False)
df2 = df2[['model', 'pred_len', 'data_id', 'mse', 'mae']]
df2['label'] = r'PDF$^\ddagger$'
df2

/tmp/ipykernel_1686659/3368251218.py:69: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,label
12,Fredformer,96,ETTm1,0.325833,0.357442,PDF$^\ddagger$
13,Fredformer,192,ETTm1,0.360031,0.379792,PDF$^\ddagger$
14,Fredformer,336,ETTm1,0.389627,0.400378,PDF$^\ddagger$
15,Fredformer,720,ETTm1,0.451406,0.437679,PDF$^\ddagger$
36,Fredformer,Avg,ETTm1,0.381724,0.393823,PDF$^\ddagger$
16,Fredformer,96,ETTm2,0.173626,0.252083,PDF$^\ddagger$
17,Fredformer,192,ETTm2,0.236756,0.294575,PDF$^\ddagger$
18,Fredformer,336,ETTm2,0.293574,0.333288,PDF$^\ddagger$
19,Fredformer,720,ETTm2,0.394318,0.392859,PDF$^\ddagger$
37,Fredformer,Avg,ETTm2,0.274569,0.318201,PDF$^\ddagger$


## analysis ablation 2

In [14]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
# aba2[(aba2['pred_len'] == 96) & (aba2['data_id'] == 'ETTm1_Random')].sort_values(by=['pred_len', 'mse'])[columns]
# aba2[(aba2['pred_len'] == 336) & (aba2['data_id'] == 'Weather_Random')].sort_values(by=['pred_len', 'mse'])[columns]
# aba2[(aba2['pred_len'] == 96) & (aba2['data_id'] == 'ETTh2_Random')].sort_values(by=['pred_len', 'mse'])[columns]
# aba2[(aba2['pred_len'] == 48) & (aba2['data_id'] == 'PEMS03_Random')].sort_values(by=['pred_len', 'mse'])[columns]
aba2[(aba2['pred_len'] == 192) & (aba2['data_id'] == 'Traffic_Random')].sort_values(by=['pred_len', 'mse'])[columns]

,model,pred_len,data_id,mse,mae,learning_rate,alpha,rank_ratio,batch_size,lradj,patience,train_epochs,reinit,use_weights
140,iTransformer,192,Traffic_Random,0.416107,0.274196,0.0005,1.0,0.6,8,type1,3,10,1,0
142,iTransformer,192,Traffic_Random,0.416709,0.274568,0.0005,1.0,0.4,8,type1,3,10,1,0
143,iTransformer,192,Traffic_Random,0.417201,0.274397,0.0005,1.0,0.8,8,type1,3,10,1,0
141,iTransformer,192,Traffic_Random,0.417333,0.275319,0.0005,1.0,0.2,8,type1,3,10,1,0


In [15]:

min_mode = 'each'

df3 = aba2.copy()
df3_m1_96 = df3[(df3['pred_len'] == 96) & (df3['data_id'] == 'ETTm1_Random') & (df3.rank_ratio == 0.4)]

df3_tra_96 = df3[(df3['pred_len'] == 96) & (df3['data_id'] == 'Traffic_Random') & (df3.rank_ratio == 0.2)]
df3_tra_192 = df3[(df3['pred_len'] == 192) & (df3['data_id'] == 'Traffic_Random') & (df3.rank_ratio == 0.2)]
df3_tra_other = df3[((df3['data_id'] == 'Traffic_Random') & (df3['pred_len'].isin([336, 720])))]

df3_wea_192 = df3[(df3['pred_len'] == 192) & (df3['data_id'] == 'Weather_Random') & (df3.rank_ratio == 0.4)]
df3_wea_336 = df3[(df3['pred_len'] == 336) & (df3['data_id'] == 'Weather_Random') & (df3.rank_ratio == 0.4)]

df3_p3 = df3[(df3['data_id'] == 'PEMS03_Random') & (df3.learning_rate == 0.0005) & (df3.rank_ratio == 0.4)]

df3_other = df3[
    ((df3['data_id'] == 'ETTm1_Random') & (df3['pred_len'].isin([192, 336, 720]))) |
    ((df3['data_id'] == 'Weather_Random') & (df3['pred_len'].isin([96, 720]))) |
    df3['data_id'].isin(['ETTh1_Random', 'ECL_Random', 'ETTh2_Random', 'ETTm2_Random', 'PEMS08_Random'])
]

df3 = pd.concat([
    df3_m1_96, 
    df3_tra_96, df3_tra_192, df3_tra_other,
    df3_wea_192, df3_wea_336,
    df3_p3,
    df3_other
], ignore_index=True)

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
if min_mode == 'group':
    df3 = df3.groupby(columns).filter(is_full_group)
    mse_mean = df3.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df3 = df3.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df3.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df3 = df3.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
df3 = df3[columns]

dst_order = ['ETTm1_Random', 'ETTm2_Random', 'ETTh1_Random', 'ETTh2_Random', 'ECL_Random', 'Traffic_Random', 'Weather_Random', 'PEMS03_Random', 'PEMS08_Random']
df3['data_id'] = pd.Categorical(df3['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df3['model'] = pd.Categorical(df3['model'], categories=model_order, ordered=True)

df3_avg = df3.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df3_avg['pred_len'] = 'Avg'

df3 = pd.concat([df3, df3_avg]).reset_index(drop=True)
df3.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)

df3.dropna(inplace=True, thresh=5)

df3['data_id'] = df3['data_id'].str.replace('_Random', '', regex=False)
df3 = df3[['model', 'pred_len', 'data_id', 'mse', 'mae']]
df3['label'] = r'PDF$^\dagger$'
df3

/tmp/ipykernel_1686659/1042507635.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df3_avg = df3.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,label
12,Fredformer,96,ETTm1,0.338156,0.365786,PDF$^\dagger$
13,Fredformer,192,ETTm1,0.369028,0.383444,PDF$^\dagger$
14,Fredformer,336,ETTm1,0.397138,0.403174,PDF$^\dagger$
15,Fredformer,720,ETTm1,0.457988,0.440628,PDF$^\dagger$
36,Fredformer,Avg,ETTm1,0.390577,0.398258,PDF$^\dagger$
16,Fredformer,96,ETTm2,0.176952,0.256114,PDF$^\dagger$
17,Fredformer,192,ETTm2,0.239129,0.296226,PDF$^\dagger$
18,Fredformer,336,ETTm2,0.301974,0.337406,PDF$^\dagger$
19,Fredformer,720,ETTm2,0.400602,0.395883,PDF$^\dagger$
37,Fredformer,Avg,ETTm2,0.279664,0.321407,PDF$^\dagger$


## analysis ablation 21

In [11]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
# aba21[(aba21['data_id'] == 'Weather_Random')].sort_values(by=['pred_len', 'mse'])[columns].round(3)
# aba21[(aba21['pred_len'] == 720) & (aba21['data_id'] == 'Weather_Random')].sort_values(by=['pred_len', 'mse'])[columns]
# aba21[(aba21['pred_len'] == 96) & (aba21['data_id'] == 'ETTm2_Random')].sort_values(by=['pred_len', 'mse'])[columns]
# aba21[(aba21['pred_len'] == 720) & (aba21['data_id'] == 'ETTh2_Random')].sort_values(by=['pred_len', 'mse'])[columns]
aba21[(aba21['pred_len'] == 192) & (aba21['data_id'] == 'Traffic_Random')].sort_values(by=['pred_len', 'mse'])[columns]
# aba21[(aba21['pred_len'] == 48) & (aba21['data_id'] == 'PEMS03_Random')].sort_values(by=['pred_len', 'mse'])[columns]
# aba21[(aba21['pred_len'] == 48) & (aba21['data_id'] == 'PEMS08_Random')].sort_values(by=['pred_len', 'mse'])[columns]

,model,pred_len,data_id,mse,mae,learning_rate,alpha,rank_ratio,batch_size,lradj,patience,train_epochs,reinit,use_weights
158,iTransformer,192,Traffic_Random,0.411706,0.273783,0.0010,1.0,1.0,8,type1,3,10,1,0
162,iTransformer,192,Traffic_Random,0.412511,0.271940,0.0007,1.0,1.0,8,type1,3,10,1,0
159,iTransformer,192,Traffic_Random,0.414683,0.272944,0.0006,1.0,1.0,8,type1,3,10,1,0
157,iTransformer,192,Traffic_Random,0.416522,0.274033,0.0005,1.0,1.0,8,type1,3,10,1,0
163,iTransformer,192,Traffic_Random,0.419722,0.276278,0.0004,1.0,1.0,8,type1,3,10,1,0
160,iTransformer,192,Traffic_Random,0.424164,0.279710,0.0003,1.0,1.0,8,type1,3,10,1,0
161,iTransformer,192,Traffic_Random,0.432910,0.286453,0.0002,1.0,1.0,8,type1,3,10,1,0


In [12]:

min_mode = 'each'

df31 = aba21.copy()
df31_m1 = df31[(df31['data_id'] == 'ETTm1_Random') & df31.learning_rate.isin([0.0005])]

df31_m2 = df31[(df31['data_id'] == 'ETTm2_Random') & df31.learning_rate.isin([0.001])]

df31_h1 = df31[(df31['data_id'] == 'ETTh1_Random') & df31.learning_rate.isin([0.0005])]

df31_h2_96 = df31[(df31['pred_len'] == 96) & (df31['data_id'] == 'ETTh2_Random') & (df31.learning_rate == 0.0005)]
df31_h2_192 = df31[(df31['pred_len'] == 192) & (df31['data_id'] == 'ETTh2_Random') & (df31.learning_rate == 0.0005)]
df31_h2_336 = df31[(df31['pred_len'] == 336) & (df31['data_id'] == 'ETTh2_Random') & (df31.learning_rate == 0.0004)]
df31_h2_720 = df31[(df31['pred_len'] == 720) & (df31['data_id'] == 'ETTh2_Random') & (df31.learning_rate == 0.0002)]

df31_ecl = df31[(df31['data_id'] == 'ECL_Random') & df31.learning_rate.isin([0.0005])]

df31_tra_96 = df31[(df31['pred_len'] == 96) & (df31['data_id'] == 'Traffic_Random') & (df31.learning_rate == 0.0004)]
df31_tra_192 = df31[(df31['pred_len'] == 192) & (df31['data_id'] == 'Traffic_Random') & (df31.learning_rate == 0.0004)]
df31_tra_other = df31[(df31['data_id'] == 'Traffic_Random') & (df31.learning_rate == 0.0005) & (df31.pred_len.isin([336, 720]))]

df31_wea_96 = df31[(df31['pred_len'] == 96) & (df31['data_id'] == 'Weather_Random') & (df31.learning_rate == 0.0002)]
df31_wea_192 = df31[(df31['pred_len'] == 192) & (df31['data_id'] == 'Weather_Random') & (df31.learning_rate == 0.0002)]
df31_wea_336 = df31[(df31['pred_len'] == 336) & (df31['data_id'] == 'Weather_Random') & (df31.learning_rate == 0.0001)]
df31_wea_720 = df31[(df31['pred_len'] == 720) & (df31['data_id'] == 'Weather_Random') & (df31.learning_rate == 0.0002)]

df31_p3_12 = df31[(df31['pred_len'] == 12) & (df31['data_id'] == 'PEMS03_Random') & (df31.learning_rate == 0.0002)]
df31_p3_24 = df31[(df31['pred_len'] == 24) & (df31['data_id'] == 'PEMS03_Random') & (df31.learning_rate == 0.0002)]
df31_p3_36 = df31[(df31['pred_len'] == 36) & (df31['data_id'] == 'PEMS03_Random') & (df31.learning_rate == 0.002)]
df31_p3_48 = df31[(df31['pred_len'] == 48) & (df31['data_id'] == 'PEMS03_Random') & (df31.learning_rate == 0.0004)]

df31_p8 = df31[(df31['data_id'] == 'PEMS08_Random') & (df31.learning_rate == 0.0005)]

df31 = pd.concat([
    df31_m1, 
    df31_m2,
    df31_h1, 
    df31_h2_96, df31_h2_192, df31_h2_336, df31_h2_720,
    df31_ecl, 
    df31_tra_96, df31_tra_192, df31_tra_other,
    df31_wea_96, df31_wea_192, df31_wea_336, df31_wea_720,
    df31_p3_12, df31_p3_24, df31_p3_36, df31_p3_48, 
    df31_p8
], ignore_index=True)

# df31_m1_96 = df31[(df31['pred_len'] == 96) & (df31['data_id'] == 'ETTm1_Random') & df31.index.isin([159])]
# df31_wea_192 = df31[(df31['pred_len'] == 192) & (df31['data_id'] == 'Weather_Random') & df31.index.isin([387])]
# df31_wea_336 = df31[(df31['pred_len'] == 336) & (df31['data_id'] == 'Weather_Random') & df31.index.isin([101])]
# df31_other = df31[
#     ((df31['data_id'] == 'ETTm1_Random') & (df31['pred_len'].isin([192, 336, 720]))) |
#     ((df31['data_id'] == 'Weather_Random') & (df31['pred_len'].isin([96, 720]))) |
#     df31['data_id'].isin(['ETTh1_Random', 'ECL_Random'])
# ]
# df31 = pd.concat([
#     df31_m1_96, 
#     df31_wea_192, df31_wea_336,
#     df31_other
# ], ignore_index=True)

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
if min_mode == 'group':
    df31 = df31.groupby(columns).filter(is_full_group)
    mse_mean = df31.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df31 = df31.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df31.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df31 = df31.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
df31 = df31[columns]

dst_order = ['ETTm1_Random', 'ETTm2_Random', 'ETTh1_Random', 'ETTh2_Random', 'ECL_Random', 'Traffic_Random', 'Weather_Random', 'PEMS03_Random', 'PEMS08_Random']
df31['data_id'] = pd.Categorical(df31['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df31['model'] = pd.Categorical(df31['model'], categories=model_order, ordered=True)

df31_avg = df31.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df31_avg['pred_len'] = 'Avg'

df31 = pd.concat([df31, df31_avg]).reset_index(drop=True)
df31.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)

df31.dropna(inplace=True, thresh=5)

df31['data_id'] = df31['data_id'].str.replace('_Random', '', regex=False)
df31 = df31[['model', 'pred_len', 'data_id', 'mse', 'mae']]
df31['label'] = r'PDF$^\star$'
df31

/tmp/ipykernel_1686659/3579893044.py:79: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df31_avg = df31.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,label
12,Fredformer,96,ETTm1,0.325554,0.354537,PDF$^\star$
13,Fredformer,192,ETTm1,0.368555,0.382219,PDF$^\star$
14,Fredformer,336,ETTm1,0.396452,0.403463,PDF$^\star$
15,Fredformer,720,ETTm1,0.457769,0.440154,PDF$^\star$
36,Fredformer,Avg,ETTm1,0.387083,0.395093,PDF$^\star$
16,Fredformer,96,ETTm2,0.177329,0.256646,PDF$^\star$
17,Fredformer,192,ETTm2,0.237759,0.295308,PDF$^\star$
18,Fredformer,336,ETTm2,0.303663,0.338213,PDF$^\star$
19,Fredformer,720,ETTm2,0.400809,0.393968,PDF$^\star$
37,Fredformer,Avg,ETTm2,0.279890,0.321034,PDF$^\star$


## analysis ablation 3

In [ ]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']

aba3[(aba3['pred_len'] == 720) & (aba3['data_id'] == 'ETTm2_PCA')].sort_values(by=['pred_len', 'mse'])[columns].head(10)

In [9]:
min_mode = 'each'

df4 = aba3.copy()

columns = ['model', 'data_id', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
if min_mode == 'group':
    df4 = df4.groupby(columns).filter(is_full_group)
    mse_mean = df4.groupby(columns)['mse'].mean().reset_index()
    idx = mse_mean.groupby(['model', 'data_id'])['mse'].idxmin()
    best_model = mse_mean.loc[idx]
    df4 = df4.merge(best_model[columns], on=columns, how='inner')
elif min_mode == 'each':
    min_mse_idx = df4.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
    df4 = df4.loc[min_mse_idx]

columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'alpha', 'rank_ratio', 'batch_size', 'lradj', 'patience', 'train_epochs', 'reinit', 'use_weights']
df4 = df4[columns]

dst_order = ['ETTm1_PCA', 'ETTm2_PCA', 'ETTh1_PCA', 'ETTh2_PCA', 'ECL_PCA', 'Traffic_PCA', 'Weather_PCA', 'PEMS03_PCA', 'PEMS08_PCA']
df4['data_id'] = pd.Categorical(df4['data_id'], categories=dst_order, ordered=True)

model_order = ['Fredformer', 'FBM_L', 'iTransformer', 'FreTS', 'TimesNet', 'MICN', 'TiDE', 'DLinear', 'DLinear_Ind', 'FEDformer', 'Autoformer', 'Transformer', 'TCN']
df4['model'] = pd.Categorical(df4['model'], categories=model_order, ordered=True)

df4_avg = df4.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df4_avg['pred_len'] = 'Avg'

df4 = pd.concat([df4, df4_avg]).reset_index(drop=True)
df4.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)

df4.dropna(inplace=True, thresh=5)

df4['data_id'] = df4['data_id'].str.replace('_PCA', '', regex=False)
df4 = df4[['model', 'pred_len', 'data_id', 'mse', 'mae']]
df4['label'] = 'PDF'
df4

/tmp/ipykernel_1686659/3316703739.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df4_avg = df4.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae,label
12,Fredformer,96,ETTm1,0.321138,0.357436,PDF
13,Fredformer,192,ETTm1,0.359581,0.378119,PDF
14,Fredformer,336,ETTm1,0.389238,0.399987,PDF
15,Fredformer,720,ETTm1,0.447020,0.434891,PDF
36,Fredformer,Avg,ETTm1,0.379244,0.392608,PDF
16,Fredformer,96,ETTm2,0.172235,0.251189,PDF
17,Fredformer,192,ETTm2,0.235445,0.293936,PDF
18,Fredformer,336,ETTm2,0.293289,0.332727,PDF
19,Fredformer,720,ETTm2,0.387706,0.389056,PDF
37,Fredformer,Avg,ETTm2,0.272169,0.316727,PDF


## concat analysis

In [16]:
compare_columns = ['pred_len', 'mse', 'mae', 'label']
aba_show = pd.concat([df1, df31[compare_columns], df3[compare_columns], df2[compare_columns], df4[compare_columns]], axis=1)


aba_res = pd.concat([df1, df31, df3, df2, df4], axis=0)
save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
# aba_res.round(3)[['data_id', 'pred_len', 'mse', 'mae', 'label']].to_csv(f'{save_root}/aba_res2.csv', index=False, float_format='%.3f')
aba_res.replace({'pred_len': {12: 96, 24: 192, 36: 336, 48: 720}}, inplace=True)

res1 = aba_res[(aba_res['data_id'].isin(['ETTh1', 'ETTh2', 'ETTm1', 'ETTm2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']))].copy()
# res1 = aba_res[(aba_res['data_id'].isin(['ETTh2', 'ETTm2', 'Traffic', 'PEMS03', 'PEMS08']))].copy()
dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Traffic', 'Weather', 'PEMS03', 'PEMS08']
res1['data_id'] = pd.Categorical(res1['data_id'], categories=dst_order, ordered=True)

label_order = ['DF', r'PDF$^\star$', r'PDF$^\dagger$', r'PDF$^\ddagger$', 'PDF']
res1['label'] = pd.Categorical(res1['label'], categories=label_order, ordered=True)

res1.sort_values(by=['label', 'data_id', 'pred_len'], inplace=True)

res1 = res1.set_index(['label', 'data_id', 'pred_len']).unstack('pred_len').swaplevel(axis=1)
columns = []
for pl in res1.columns.levels[0]:
    columns.append((pl, 'mse'))
    columns.append((pl, 'mae'))
res1 = res1[columns]


save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
res1.round(3).to_csv(f'{save_root}/aba_res_row2_full.csv', float_format='%.3f')

res1.round(3).to_latex(f'{save_root}/aba_res_row2_full.tex', float_format='%.3f', escape=False, index=True, header=True)


res1


pred_len                      96                 192                 336  \
                             mse       mae       mse       mae       mse   
label          data_id                                                     
DF             ETTm1    0.326369  0.360869  0.365194  0.382132  0.395987   
               ETTm2    0.177032  0.259926  0.241639  0.299673  0.301849   
               ETTh1    0.377193  0.395888  0.437019  0.425390  0.485769   
               ETTh2    0.293376  0.343882  0.371951  0.391174  0.420394   
               ECL      0.150035  0.241510  0.168114  0.259067  0.182346   
               Traffic  0.396554  0.271166  0.415822  0.278879  0.429438   
               Weather  0.173683  0.227737  0.212846  0.266092  0.270492   
               PEMS03   0.096175  0.216982  0.094863  0.209643  0.106963   
               PEMS08   0.083674  0.187030  0.123175  0.226933  0.169713   
PDF$^\star$    ETTm1    0.325554  0.354537  0.368555  0.382219  0.396452   
               ETTm2    0.177329  0.256646  0.237759  0.295308  0.303663   
               ETTh1    0.375840  0.394460  0.436310  0.429147  0.478609   
               ETTh2    0.289142  0.334434  0.366610  0.383627  0.414149   
               ECL      0.149972  0.239036  0.165589  0.254157  0.179019   
               Traffic  0.397707  0.266075  0.419722  0.276278  0.429927   
               Weather  0.175259  0.220235  0.213588  0.256826  0.268914   
               PEMS03   0.087124  0.198626  0.107432  0.220973  0.107496   
               PEMS08   0.083269  0.184204  0.122661  0.223602  0.164709   
PDF$^\dagger$  ETTm1    0.338156  0.365786  0.369028  0.383444  0.397138   
               ETTm2    0.176952  0.256114  0.239129  0.296226  0.301974   
               ETTh1    0.376181  0.395312  0.436743  0.430490  0.478147   
               ETTh2    0.287725  0.334604  0.368929  0.384772  0.404860   
               ECL      0.149824  0.238775  0.164379  0.252756  0.177527   
               Traffic  0.395188  0.265476  0.417333  0.275319  0.429706   
               Weather  0.170403  0.215875  0.213272  0.259083  0.262421   
               PEMS03   0.078384  0.189765  0.089869  0.203921  0.107705   
               PEMS08   0.083181  0.184152  0.122851  0.222835  0.164561   
PDF$^\ddagger$ ETTm1    0.325833  0.357442  0.360031  0.379792  0.389627   
               ETTm2    0.173626  0.252083  0.236756  0.294575  0.293574   
               ETTh1    0.373171  0.395279  0.432823  0.422560  0.475918   
               ETTh2    0.282361  0.333281  0.362798  0.381201  0.394194   
               ECL      0.147042  0.238406  0.161697  0.252440  0.174402   
               Traffic  0.393947  0.267404  0.411923  0.275569  0.423960   
               Weather  0.171810  0.220203  0.210615  0.258574  0.260762   
               PEMS03   0.071650  0.178306  0.090149  0.201523  0.107319   
               PEMS08   0.082110  0.183141  0.119759  0.221874  0.162443   
PDF            ETTm1    0.321138  0.357436  0.359581  0.378119  0.389238   
               ETTm2    0.172235  0.251189  0.235445  0.293936  0.293289   
               ETTh1    0.368003  0.390765  0.424089  0.421955  0.466960   
               ETTh2    0.281609  0.330220  0.359048  0.380537  0.393813   
               ECL      0.144906  0.234782  0.158948  0.248666  0.173076   
               Traffic  0.392572  0.264985  0.409847  0.274966  0.420893   
               Weather  0.169180  0.218550  0.210151  0.257542  0.258569   
               PEMS03   0.069891  0.176210  0.087064  0.198136  0.105463   
               PEMS08   0.081086  0.183092  0.117261  0.217823  0.156879   

pred_len                               720                 Avg            
                             mae       mse       mae       mse       mae  
label          data_id                                                    
DF             ETTm1    0.404369  0.459217  0.444342  0.386692  0.397928  
               ETTm2    0.340199  0.398819  0.39673

In [17]:
import pandas as pd
import numpy as np

def wrap_min_second(series):
    # 格式化为三位小数字符串
    formatted = series.apply(lambda x: "{:.3f}".format(x))
    # 按值排序获得索引
    sorted_idx = series.argsort()
    min_idx = sorted_idx[0]
    formatted.iloc[min_idx] = f"\\bst{{{formatted.iloc[min_idx]}}}"
    if len(sorted_idx) > 1:
        second_idx = sorted_idx[1]
        formatted.iloc[second_idx] = f"\\subbst{{{formatted.iloc[second_idx]}}}"
    return formatted

def mark_res1(res1):
    marked = res1.copy()
    for data_id in res1.index.levels[1]:
        for pred_len in res1.columns.levels[0]:
            for metric in ['mse', 'mae']:
                col = (pred_len, metric)
                vals = res1.xs(data_id, level='data_id')[col]
                marked_col = wrap_min_second(vals)
                marked.loc[(slice(None), data_id), col] = marked_col.values
    return marked

def to_latex(marked):
    # 输出 latex, a{}和b{}不转义
    return marked.reset_index().to_latex(index=False, escape=False)

# 使用方法：
marked = mark_res1(res1)
latex_str = to_latex(marked)
with open(f'{save_root}/aba_res_row2_full.tex', 'w') as f:
    f.write(latex_str)

/tmp/ipykernel_1686659/787871132.py:9: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  min_idx = sorted_idx[0]
/tmp/ipykernel_1686659/787871132.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  second_idx = sorted_idx[1]
/tmp/ipykernel_1686659/787871132.py:24: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['0.326' '\\subbst{0.326}' '0.338' '0.326' '\\bst{0.321}']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  marked.loc[(slice(None), data_id), col] = marked_col.values
/tmp/ipykernel_1686659